# **VMC Simulation of Superfluid 4He**
### QMC Final Project

By: **Joaquín Gabriel Márquez Olguín**

Course 2025/26

**Goal:** Determine the ground state energy and structural properties of Superfluid Helium-4 in a 3D box with Periodic Boundary Conditions (PBC) using the Variational Monte Carlo (VMC) method. We will model the interatomic interactions using the Aziz II potential and optimize a Jastrow-type trial wave function.

**Physical System**

We consider a system of $N$ Helium-4 atoms (bosons) in a cubic box of volume $V = L^3$ at density $\rho = N/V$.The Hamiltonian is:$$\hat{H} = -\frac{\hbar^2}{2m} \sum_{i=1}^{N} \nabla_i^2 + \sum_{i<j}^{N} V(r_{ij})$$

The Hamiltonian of the system is:

$$\hat{H} = -\frac{\hbar^2}{2m} \sum_{i=1}^{N} \nabla_i^2 + \sum_{i<j}^{N} V(r_{ij}).$$

* The **Interaction Potential** $V(r_{ij})$ is given by the Aziz II potential (proposed by Aziz et al.), which accurately captures the He-He interaction. It takes the form:

    $$V(r) = \epsilon \left[ A \exp(-\alpha_p x +\beta x^2) - F(x) \left( \frac{C_6}{x^6} + \frac{C_8}{x^8} + \frac{C_{10}}{x^{10}} \right)\right],$$

    where $x = r / r_m$, and the damping function $F(x)$ handles the short-range behavior:
    
    $$
    F(x) = \begin{cases} 
    \exp \left[ -\left( \frac{D}{x} - 1 \right)^2 \right] & \text{if } x < D \\
    1 & \text{if } x \ge D
    \end{cases}
    $$

    The parameters are given in the following table:


    | Parameter | Value | Unit | Description |
    | :--- | :--- | :--- | :--- |
    | **$r_m$** | 2.98 | Å | Minimum potential distance |
    | **$D$** | 1.24 | - | Damping parameter |
    | **$A$** | $0.554 \times 10^6$ | K | Repulsive amplitude |
    | **$\alpha_p$** | 13.35 | - | Repulsive decay constant |
    | **$\beta_p$** | 13.35 | - | ? |
    | **$C_6$** | 1.373 | K·Å⁶ | Dispersion coefficient |
    | **$C_8$** | 0.425 | K·Å⁸ | Dispersion coefficient |
    | **$C_{10}$** | 0.178 | K·Å¹⁰ | Dispersion coefficient |
    | **$\epsilon$** | 10.8 | K | Well depth |

$\vspace{1cm}$


**Implementation Details:**

* To simulate a bulk superfluid system using a finite number of particles $N$, we employ **Periodic Boundary Conditions (PBC)** in all three Cartesian directions:

    * The particles are confined to a cubic box of side length $L = (N/\rho)^{1/3}$.
    * When calculating the distance $r_{ij}$ between any pair of particles $i$ and $j$, we consider the shortest distance between particle $i$ and any periodic image of particle $j$:
        $$r_{ij} = \min_{\mathbf{n} \in \mathbb{Z}^3} |\mathbf{r}_i - \mathbf{r}_j - \mathbf{n}L|$$
        
        This ensures that the potential energy and trial wave function respects the periodicity of the box.

* We utilize a variational approach with a **trial wave function** $\Psi_T$ of the Jastrow pair-product form (which captures the strong two-body correlations induced by the hard core of the Helium potential):

    $$\Psi_T(\mathbf{R}) = \prod_{i<j}^{N} \exp \left( -\frac{\alpha}{|r_{ij}|^{\beta}} \right),$$

    where $\alpha$ and $\beta$ in the project description are the variational parameters we will optimize to minimize the total energy.

**VMC and Energy Estimation:**

We use the Metropolis-Hastings algorithm to sample configurations $\mathbf{R}$ distributed according to the probability density $P(\mathbf{R}) = \frac{|\Psi_T(\mathbf{R})|^2}{\int |\Psi_T|^2 d\mathbf{R}}$.

To evaluate the ground state energy, we calculate the *Local Energy* $E_L(\mathbf{R})$ averaged over the sampled configurations. As we have seen, this estimator has the property that if $\Psi_T$ is the exact eigenstate, $E_L$ is constant and equal to the exact energy (zero variance property).

$$\langle E \rangle = \frac{1}{M} \sum_{k=1}^{M} E_L(\mathbf{R}_k)$$

The Local Energy is defined as:

$$E_L(\mathbf{R}) = \frac{\hat{H} \Psi_T(\mathbf{R})}{\Psi_T(\mathbf{R})} = -\frac{\hbar^2}{2m} \sum_{i=1}^N \frac{\nabla_i^2 \Psi_T}{\Psi_T} + V(\mathbf{R})$$

For the Jastrow form $\Psi = e^{-U}$ (where $U = \sum u(r_{ij})$), it is computationally efficient to split the kinetic term into a gradient squared term and a Laplacian term:

$$-\frac{\hbar^2}{2m} \frac{\nabla_i^2 \Psi_T}{\Psi_T} = -\frac{\hbar^2}{2m} \left[ \nabla_i^2 (-\ln \Psi_T) + (\nabla_i \ln \Psi_T)^2 \right]$$

This approach avoids numerical instabilities and allows us to monitor both the Kinetic and Potential contributions separately during the simulation (to better reafirm our simulation results).

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
import time

# POTENTIAL PARAMETERS

RM = 2.98
D_DAMP = 1.24
A_REP = 0.554e6
ALPHA_REP = 13.35
BETA_REP = 0.0
C6 = 1.373
C8 = 0.425
C10 = 0.178
EPSILON = 10.8

# HELPER FUNCTIONS

@njit
def pbc_dist(r1, r2, L):
    """
    Calculates the vector and scalar distance r_ij between two particles
    under Periodic Boundary Conditions
    """
    dx = r1[0] - r2[0]
    dy = r1[1] - r2[1]
    dz = r1[2] - r2[2]
    
    # Minimum Image Convention
    dx -= L * round(dx / L)
    dy -= L * round(dy / L)
    dz -= L * round(dz / L)

    r_vec = np.array([dx, dy, dz])
    r_scalar = np.sqrt(dx*dx + dy*dy + dz*dz)
    
    return r_vec, r_scalar

@njit
def psi_T(positions, L, alpha, beta):
    """
    Many-body trial wave function for N particles:
    Psi = Product_{i<j} exp( - alpha / r_ij^beta )
    """
    N = len(positions)
    exp_term = 0.0
    
    for i in range(N):
        for j in range(i + 1, N):
            _, r = pbc_dist(positions[i], positions[j], L)

            if r > 1e-12:
                exp_term -= (alpha / (r**beta))
            else:
                return 0.0 # null overlap at large distances
                
    return np.exp(exp_term)

@njit
def log_probability(positions, L, alpha, beta):
    """
    Computes ln(|Psi|^2) = 2 * ln(Psi) 
                         = -2 * sum_{i<j} (alpha / r_ij^beta)
    """
    N = len(positions)
    log_prob = 0.0
    
    for i in range(N):
        for j in range(i + 1, N):
            _, r = pbc_dist(positions[i], positions[j], L)
            
            if r < 1e-12: 
                return -1e20 # Hard core overlap
            
            u = alpha / (r**beta)
            log_prob -= 2.0 * u
            
    return log_prob

# ENERGY + POTENTIAL

HBAR2_2M = 6.0596   # K * A^2 (hbar^2/2m_He)
RHO_HE = 0.0218     # Equilibrium density of He4 (atoms/A^3)

@njit
def local_energy(positions, L, alpha_var, beta_var):
    """
    Calculates Local Energy E_L = T_loc + V_loc
    T_loc = - (hbar^2/2m) * [ Sum_i (Lap_i ln Psi + |Grad_i ln Psi|^2) ]
    V_loc = Aziz II Potential
    """
    N = len(positions)
    
    potential_energy = 0.0
    kinetic_laplacian = 0.0
    kinetic_grad_sq = 0.0
    
    gradients = np.zeros((N, 3)) 
    
    # Loop over pairs
    for i in range(N):
        for j in range(i + 1, N):

            r_vec, _ = pbc_dist(positions[i], positions[j], L)
            r = np.linalg.norm(r_vec)
            
            if r < 1e-12: continue # Avoid div by zero
            
            # Aziz II PPOTENTIAL
            RM = 2.98
            D_DAMP = 1.24
            A_REP = 0.554e6
            ALPHA_REP = 13.35
            BETA_REP = 0.0
            C6 = 1.373
            C8 = 0.425
            C10 = 0.178
            EPSILON = 10.8

            x = r / RM
            # Damping F(x)
            if x < D_DAMP:
                F = np.exp(-(D_DAMP/x - 1)**2)
            else:
                F = 1.0
            
            V_pair = EPSILON * ( A_REP * np.exp(-ALPHA_REP * x +BETA_REP * x**2) - 
                                F * (C6/x**6 + C8/x**8 + C10/x**10) )
            potential_energy += V_pair
            
            # KINETIC PART: LAPLACIAN + GRADIENT ESTIMATORS
            # u(r) = alpha / r^beta
            # ln Psi_T = - sum u(r)
            
            # u = alpha * r^(-beta)
            u_val = alpha_var * (r**(-beta_var))
            
            # u' = -beta * alpha * r^(-beta-1) = -beta * u / r
            du_dr = -beta_var * u_val / r
            
            # u'' = -beta * (-beta-1) * alpha * r^(-beta-2)
            #     = beta * (beta+1) * u / r^2
            d2u_dr2 = beta_var * (beta_var + 1) * u_val / (r**2)
            
            # Laplacian_i (ln Psi): 
            #   Lap(-u) = - (u'' + 2/r * u')
            lap_term = - (d2u_dr2 + (2.0/r)*du_dr)
            
            # Both particles i and j feel this laplacian scalar
            kinetic_part_laplacian += 2.0 * lap_term 
            
            # Gradient_i (ln Psi):
            #   Grad_i (-u) = -u' * (r_vec / r)
            grad_contribution = -du_dr * (r_vec / r)
            
            # Add vector contribution to i, subtract from j
            gradients[i] += grad_contribution
            gradients[j] -= grad_contribution
            
    # Sum squares of total gradients for each particle
    for i in range(N):
        kinetic_part_grad_sq += np.sum(gradients[i]**2)
        
    kinetic_energy = -HBAR2_2M * (kinetic_part_laplacian + kinetic_part_grad_sq)
    
    return kinetic_energy + potential_energy



# VMC ALGORITHM

@njit
def VMC_simulation(n_steps, N, alpha, beta, delta):
    """
    Runs Variational Monte Carlo simulation for Superfluid Helium.

    Inputs:
        n_steps        : number of MC steps to perform
        N              : number of particles
        alpha, beta    : variational parameters for the trial wavefunction
        delta          : maximum displacement for trial moves
    Outputs:
        energies_lap   : energy history (Laplacian estimator)
        energies_grad  : energy history (Drift Force/Gradient estimator)
        acceptance_rate: acceptance rate per step
        x_history      : history particle positions (N_steps, N, 3)
    """
    # 3D Simulation Box
    L = (N / RHO_HE)**(1.0/3.0)
    
    # Storage for results
    # We will sample energy less frequently to save time
    sample_interval = 10 
    n_samples = (n_steps // sample_interval) + 1

    energies_lap = np.zeros(n_samples)
    energies_grad = np.zeros(n_samples)
    x_history = np.zeros((n_samples, N, 3)) 

    # Initialized random positions in Box L
    x = np.random.rand(N, 3) * L

    accepted_moves = 0
    attempted_moves = 0

    # Calculate current (log) probability
    log_P_current = log_probability(x, L, alpha, beta)

    # MC loop
    sample_idx = 0
    for step in range(n_steps):
        # Propose a random move to just one random particle
        i_particle = np.random.randint(N)

        old_pos_i = x[i_particle].copy()

        dx = delta * (2.0 * np.random.rand(3) - 1.0)
        x[i_particle] += dx
        
        # Apply PBC to proposed positions
        for k in range(3):
            x[i_particle, k] -= L * np.floor(x[i_particle, k] / L)

        # Compute new (log) probability
        log_P_new = log_probability(x, L, alpha, beta)

        # Metropolis algorithm (Log version to avoind numerical instability)
        # Ratio A = P_new / P_old  ==>  ln(A) = ln(P_new) - ln(P_old)
        if (log_P_new - log_P_current) > np.log(np.random.rand()):
            # We keep x as it is now
            log_P_current = log_P_new
            accepted_moves += 1
        else:
            # Reject move (go back to old position)
            x[i_particle] = old_pos_i
        
        attempted_moves += 1
        
        # Measure Local Energy
        if step % sample_interval == 0:
            e_lap, e_grad = local_energy(x, L, alpha, beta)
        
            energies_lap[sample_idx] = e_lap
            energies_grad[sample_idx] = e_grad
            x_history[sample_idx, :, :] = x
            sample_idx += 1
        
    return energies_lap, energies_grad, accepted_moves / attempted_moves, x_history

Now, we compute the Ground State (GS) Energy by finding the optimal varioational parameteres ($\alpha$, $\beta$) that minimize the expectation value of the Hamiltonian.

In order to do it, we perform a Grid search over the paramters to find the minimum energy.

Some implementation details:
* We discard the first 20-30% of steps to allow the system to equilibrate from the random initial configuration.
* We calculate the mean energy and the statistical error.
* We loop over a range of $\alpha$ and $\beta$ values.

In [13]:
# --- 1. PHYSICAL CONSTANTS & PARAMETERS ---
HBAR2_2M = 6.0596   # K * A^2 (hbar^2/2m_He)
RHO_HE = 0.0218     # Equilibrium density (atoms/A^3)

# Aziz Potential Parameters
RM = 2.98
D_DAMP = 1.24
A_REP = 0.554e6
ALPHA_REP = 13.35
BETA_REP = 0.0
C6 = 1.373
C8 = 0.425
C10 = 0.178
EPSILON = 10.8

# SIMULATION PARAMETERS
N_PARTICLES = 32         # 32 is standard (64 is better for production)
DENSITY = 0.0218         # Equilibrium density of He-4 (atoms/A^3)
N_STEPS = 50000          # Increased to 50k because single-particle moves need more steps
THERMALIZATION = 5000    # Steps to discard (must be < N_STEPS)
DELTA = 0.4    

          # Metropolis step size

# Important: Must match the interval defined inside VMC_simulation
SAMPLE_INTERVAL = 10     

# --- VARIATIONAL PARAMETER GRID ---
alpha_list = np.linspace(2.6, 3.2, 5)  
beta_list = [5.0]                    

print(f"--- STARTING SIMULATION ---")
print(f"System: {N_PARTICLES} Helium atoms at density {DENSITY}")
print(f"Steps: {N_STEPS} (Single Particle Moves)")
print(f"Grid Search: {len(alpha_list)} alpha values x {len(beta_list)} beta values")
print("-" * 60)

results = []

for beta in beta_list:
    for alpha in alpha_list:
        start_time = time.time()
        
        # 1. RUN VMC
        E_lap_history, E_grad_history, acc_rate, _ = VMC_simulation(
            N_STEPS, N_PARTICLES, alpha, beta, DELTA
        )
        
        # 2. ANALYZE DATA
        # Correcting index for the sampling interval
        therm_idx = THERMALIZATION // SAMPLE_INTERVAL
        
        if therm_idx >= len(E_lap_history):
            print(f"Warning: Thermalization {THERMALIZATION} > Data Size. Reducing to 50%.")
            therm_idx = len(E_lap_history) // 2
            
        valid_energies = E_lap_history[therm_idx:]
        
        # Calculate Energy PER PARTICLE
        E_total_mean = np.mean(valid_energies)
        E_per_particle = E_total_mean / N_PARTICLES
        
        # Calculate Error
        # Note: We divide by sqrt(N_samples) to get standard error of the mean
        # We divide by N_PARTICLES to get error *per particle*
        E_error = np.std(valid_energies) / np.sqrt(len(valid_energies)) / N_PARTICLES
        
        elapsed = time.time() - start_time
        
        # 3. STORE AND PRINT
        print(f"Alpha: {alpha:.2f} | Lambda: {beta:.2f} | "
              f"E/N: {E_per_particle:.4f} +/- {E_error:.4f} K | "
              f"Acc: {acc_rate:.1%} | Time: {elapsed:.2f}s")
        
        results.append((alpha, beta, E_per_particle, E_error))

# --- FIND BEST PARAMETERS ---
results = np.array(results)
best_idx = np.argmin(results[:, 2]) # Index of minimum Energy
best_alpha = results[best_idx, 0]
best_lmbda = results[best_idx, 1]
min_energy = results[best_idx, 2]
min_err = results[best_idx, 3]

print("-" * 60)
print(f"OPTIMAL RESULT FOUND:")
print(f"Best Alpha:  {best_alpha:.2f}")
print(f"Best Lambda: {best_lmbda:.2f}")
print(f"Ground State Energy: {min_energy:.4f} +/- {min_err:.4f} K per particle")
print("-" * 60)

# --- OPTIONAL: PLOT RESULTS ---
plt.figure(figsize=(8, 5))
subset = results[results[:, 1] == best_lmbda]

plt.errorbar(subset[:, 0], subset[:, 2], yerr=subset[:, 3], fmt='-o', capsize=5, label=f'Lambda={best_lmbda}')
plt.title(f'Ground State Energy Optimization (N={N_PARTICLES})')
plt.xlabel(r'Variational Parameter $\alpha$')
plt.ylabel('Energy per particle (K)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

--- STARTING SIMULATION ---
System: 32 Helium atoms at density 0.0218
Steps: 50000 (Single Particle Moves)
Grid Search: 5 alpha values x 1 beta values
------------------------------------------------------------


NotDefinedError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mFailed in nopython mode pipeline (step: analyzing bytecode)
[1m[1mThe compiler failed to analyze the bytecode. Variable 'kinetic_part_laplacian' is not defined.
[1m
File "C:\Users\jmarq\AppData\Local\Temp\ipykernel_13544\144851438.py", line 151:[0m
[1mdef local_energy(positions, L, alpha_var, beta_var):
    <source elided>
            # Both particles i and j feel this laplacian scalar
[1m            kinetic_part_laplacian += 2.0 * lap_term 
[0m            [1m^[0m[0m
[0m
[0m[1mDuring: Pass translate_bytecode[0m
[0m[1mDuring: resolving callee type: type(CPUDispatcher(<function local_energy at 0x0000027496AD4CC0>))[0m
[0m[1mDuring: typing of call at C:\Users\jmarq\AppData\Local\Temp\ipykernel_13544\144851438.py (242)[0m
[0m[1mDuring: Pass nopython_type_inference[0m